# テイラー展開 1：多項式で関数を近似する ―オイラーの公式まで

関数 $f$ を、点 $0$ のまわりで多項式

$$
f(x)=f(0)+f'(0)\,x+\frac{f''(0)}{2!}x^2+\frac{f'''(0)}{3!}x^3+\cdots=\sum_{k=0}^\infty\frac{f^{(k)}(0)}{k!}x^k
$$

で表すのが**マクローリン展開**（点 $a$ のまわりなら**テイラー展開**）です。このノートブックでは、

1. 次数を上げると近似が広がる様子（スライダー）
2. 近似がどこまでも広がるとは限らない例（収束半径）
3. 誤差の大きさ
4. **オイラーの公式** $e^{i\theta}=\cos\theta+i\sin\theta$ と $e^{i\pi}=-1$ を、級数から導く
5. テイラー展開できない「なめらかな関数」
6. 級数をそのまま足すと、丸め誤差で答えが崩れる例

を、計算して見ます。次のノートブックでは、同じ級数の $x$ に**行列**を入れます。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from math import factorial
from numpy.typing import NDArray
from plotly.subplots import make_subplots

Floats = NDArray[np.float64]       # 実数の配列（型ヒント用の別名）
Complexes = NDArray[np.complex128]
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなくリストを渡す（NB-02 を参照）

## 1. 次数を上げると、近似が広がる

よく使う関数のマクローリン展開です。

$$
e^x=\sum_{k\ge0}\frac{x^k}{k!},\qquad \sin x=\sum_{k\ge0}\frac{(-1)^kx^{2k+1}}{(2k+1)!},\qquad \cos x=\sum_{k\ge0}\frac{(-1)^kx^{2k}}{(2k)!}
$$

$$
\ln(1+x)=\sum_{k\ge1}\frac{(-1)^{k+1}x^k}{k},\qquad \frac1{1+x^2}=\sum_{k\ge0}(-1)^kx^{2k}
$$

関数と次数 $N$ を選び、$N$ 次までの部分和（青）が元の関数（灰色）にどう近づくかを見ます。

In [ ]:
def series_terms(name: str, x: Floats, N: int) -> Floats:
    """N 次までの各項を並べた配列（行が項、列が x の点）"""
    terms = []
    for k in range(N + 1):
        if name == "exp":
            c = 1 / factorial(k)
        elif name == "sin":
            c = 0.0 if k % 2 == 0 else (-1) ** ((k - 1) // 2) / factorial(k)
        elif name == "cos":
            c = 0.0 if k % 2 == 1 else (-1) ** (k // 2) / factorial(k)
        elif name == "log1p":
            c = 0.0 if k == 0 else (-1) ** (k + 1) / k
        else:  # "rational" = 1/(1+x²)
            c = 0.0 if k % 2 == 1 else (-1) ** (k // 2)
        terms.append(c * x**k)
    return np.array(terms)


FUNCS = {"exp": ("e^x", np.exp), "sin": ("sin x", np.sin), "cos": ("cos x", np.cos),
         "log1p": ("ln(1+x)", np.log1p), "rational": ("1/(1+x²)", lambda x: 1 / (1 + x**2))}


def partial_sum(name: str, x: Floats, N: int) -> Floats:
    return series_terms(name, x, N).sum(axis=0)


assert abs(partial_sum("sin", np.array([1.0]), 15)[0] - np.sin(1.0)) < 1e-12
assert abs(partial_sum("exp", np.array([2.0]), 30)[0] - np.exp(2.0)) < 1e-12
assert abs(partial_sum("log1p", np.array([0.5]), 60)[0] - np.log1p(0.5)) < 1e-12

xs = np.linspace(-4, 4, 400)
fig_taylor = go.FigureWidget([go.Scatter(x=xs.tolist(), y=[], mode="lines", line=dict(color="#aaa", width=5), name="元の関数"),
                              go.Scatter(x=xs.tolist(), y=[], mode="lines", line=dict(color="#1f77b4", width=2), name="N 次までの部分和")])
fig_taylor.update_layout(height=420, width=820, margin=dict(t=20), yaxis=dict(range=[-3, 3]), xaxis_title="x")
func_dd = W.Dropdown(options=[(v[0], k) for k, v in FUNCS.items()], value="sin", description="関数")
N_slider = W.IntSlider(value=5, min=0, max=30, description="次数 N")
info = W.HTML()


def update(*_: object) -> None:
    """関数と次数が変わるたびに、部分和を描き直す"""
    name, N = func_dd.value, N_slider.value
    f = FUNCS[name][1]
    with np.errstate(invalid="ignore", divide="ignore"):
        exact = f(xs)
    approx = partial_sum(name, xs, N)
    with fig_taylor.batch_update():
        fig_taylor.data[0].y = np.where(np.isfinite(exact), np.clip(exact, -50, 50), np.nan).tolist()
        fig_taylor.data[1].y = np.clip(approx, -50, 50).tolist()
    ok = np.abs(approx - exact) < 0.01
    inside = xs[ok & np.isfinite(exact)]
    span = f"誤差 0.01 以内の範囲：x ≈ {inside.min():.2f} 〜 {inside.max():.2f}" if len(inside) else "誤差 0.01 以内の点なし"
    info.value = f"{FUNCS[name][0]} の {N} 次までの部分和：{span}"


for w in (func_dd, N_slider):
    w.observe(update, "value")
update()
W.VBox([W.HBox([func_dd, N_slider]), info, fig_taylor])

**考えてみよう**

- $e^x$、$\sin x$、$\cos x$ は、$N$ を上げるほど、近似できる範囲がどこまでも広がります。
- $\ln(1+x)$ と $1/(1+x^2)$ は、$N$ をいくら上げても、$|x|<1$ の外では近似が良くなりません（むしろ暴れます）。$1/(1+x^2)$ は実数全体でなめらかなのに、なぜでしょうか（次の節）。

## 2. 収束半径：近似が届く範囲には限りがある

$x=0.5$（内側）と $x=1.5$（外側）で、次数 $N$ を上げたときの誤差を比べます。

In [ ]:
Ns = np.arange(1, 41)
fig = go.Figure()
for name in ("log1p", "rational"):
    for x0, dash in ((0.5, "solid"), (1.5, "dash")):
        err = [abs(partial_sum(name, np.array([x0]), int(N))[0] - FUNCS[name][1](np.array([x0]))[0]) for N in Ns]
        fig.add_trace(go.Scatter(x=Ns.tolist(), y=err, mode="lines", line=dict(dash=dash),
                                 name=f"{FUNCS[name][0]}、x = {x0}"))
fig.update_layout(height=380, width=800, yaxis_type="log", xaxis_title="次数 N", yaxis_title="誤差", margin=dict(t=20))

# 内側では誤差 → 0、外側では誤差が増える
assert abs(partial_sum("rational", np.array([0.5]), 40)[0] - 0.8) < 1e-10
assert abs(partial_sum("rational", np.array([1.5]), 40)[0]) > 1e6
fig

**理由（複素数の世界を見ると分かる）**：$1/(1+x^2)$ は、実数の範囲ではどこでもなめらかですが、$x$ を複素数 $z$ に広げると、$z=\pm i$ で分母が $0$ になります。マクローリン展開が届く範囲は、**複素平面で原点から一番近い「壊れる点」までの距離**（この場合 $|\pm i|=1$）で決まり、これを**収束半径**と呼びます。$\ln(1+x)$ も $x=-1$ で壊れるので、収束半径は $1$ です。$e^x,\sin x,\cos x$ はどこでも壊れないので、収束半径は無限大です。

実数だけを見ていては分からない理由が、複素数で見ると分かる。これは、学習ロードマップの候補17（複素解析）への入口です。

## 3. 誤差の大きさ：ラグランジュの剰余項

$N$ 次で打ち切ったときの誤差は、$0$ と $x$ の間のある $\xi$ を使って

$$
R_N(x)=\frac{f^{(N+1)}(\xi)}{(N+1)!}\,x^{N+1}
$$

と書けます（**ラグランジュの剰余項**）。$e^x$ なら $f^{(N+1)}(\xi)=e^\xi\le e^{|x|}$ なので、誤差は $e^{|x|}\,\frac{|x|^{N+1}}{(N+1)!}$ 以下です。$(N+1)!$ は指数関数よりずっと速く大きくなるので、誤差はやがて急速に小さくなります。

In [ ]:
x0 = 3.0
Ns = np.arange(0, 31)
err = np.array([abs(partial_sum("exp", np.array([x0]), int(N))[0] - np.exp(x0)) for N in Ns])
bound = np.array([np.exp(x0) * x0 ** (N + 1) / factorial(int(N) + 1) for N in Ns])
mask = err > 1e-12
assert np.all(err[mask] <= bound[mask] * (1 + 1e-9))       # 誤差は剰余項の上限以下
fig = go.Figure([go.Scatter(x=Ns.tolist(), y=err.tolist(), mode="lines+markers", name="実際の誤差（x = 3）"),
                 go.Scatter(x=Ns.tolist(), y=bound.tolist(), mode="lines", line=dict(dash="dash"), name="剰余項による上限")])
fig.update_layout(height=360, width=780, yaxis_type="log", xaxis_title="次数 N", yaxis_title="誤差", margin=dict(t=20))
fig

## 4. オイラーの公式：$e^{i\theta}=\cos\theta+i\sin\theta$

$e^x$ の級数の $x$ に $i\theta$ を入れます。$i^2=-1$、$i^3=-i$、$i^4=1$ と4つごとにくり返すので、項を実部と虚部に分けると

$$
e^{i\theta}=\Big(1-\frac{\theta^2}{2!}+\frac{\theta^4}{4!}-\cdots\Big)+i\Big(\theta-\frac{\theta^3}{3!}+\frac{\theta^5}{5!}-\cdots\Big)=\cos\theta+i\sin\theta
$$

で、実部が $\cos$ の級数、虚部が $\sin$ の級数そのものです。$\theta=\pi$ とすると $e^{i\pi}=\cos\pi+i\sin\pi=-1$ です。

下の図は、部分和 $S_N=\sum_{k=0}^{N}\frac{(i\theta)^k}{k!}$ を複素平面に描いたものです。項を1つ足すごとに $i$ 倍の向き（$90^\circ$ 回転）に進むので、部分和は渦を巻きながら $e^{i\theta}$（単位円の上の点）に近づきます。

In [ ]:
def euler_partial_sums(theta: float, N: int) -> Complexes:
    terms = np.array([(1j * theta) ** k / factorial(k) for k in range(N + 1)])
    return np.cumsum(terms)


# 実部 = cos の級数、虚部 = sin の級数
for theta in (0.7, np.pi, 2.5):
    S = euler_partial_sums(theta, 40)[-1]
    assert np.isclose(S.real, partial_sum("cos", np.array([theta]), 40)[0])
    assert np.isclose(S.imag, partial_sum("sin", np.array([theta]), 40)[0])
    assert np.isclose(S, np.exp(1j * theta))
print("e^(iπ) の部分和（30 次）=", np.round(euler_partial_sums(np.pi, 30)[-1], 12))

t = np.linspace(0, 2 * np.pi, 200)
fig_euler = go.FigureWidget([go.Scatter(x=np.cos(t).tolist(), y=np.sin(t).tolist(), mode="lines", line=dict(color="#ccc"), name="単位円"),
                             go.Scatter(x=[], y=[], mode="lines+markers", line=dict(color="#1f77b4"), marker=dict(size=6), name="部分和 S₀, S₁, …"),
                             go.Scatter(x=[], y=[], mode="markers", marker=dict(size=14, color="#d62728", symbol="x"), name="e^(iθ)")])
fig_euler.update_layout(height=520, width=620, margin=dict(t=20), xaxis=dict(range=[-4, 2.5], title="実部"),
                        yaxis=dict(range=[-3, 3.5], scaleanchor="x", title="虚部"))
theta_s = W.FloatSlider(value=180, min=0, max=360, step=5, description="θ [度]")
N_s = W.IntSlider(value=8, min=0, max=30, description="次数 N")
info2 = W.HTML()


def update_euler(*_: object) -> None:
    """θ と次数が変わるたびに、部分和の道を描き直す"""
    theta, N = np.radians(theta_s.value), N_s.value
    S = euler_partial_sums(theta, N)
    target = np.exp(1j * theta)
    with fig_euler.batch_update():
        fig_euler.data[1].x, fig_euler.data[1].y = S.real.tolist(), S.imag.tolist()
        fig_euler.data[2].x, fig_euler.data[2].y = [float(target.real)], [float(target.imag)]
    info2.value = (f"S_{N} = {S[-1].real:+.6f} {S[-1].imag:+.6f} i、e^(iθ) = {target.real:+.6f} {target.imag:+.6f} i、"
                   f"差 = {abs(S[-1] - target):.2e}")


for w in (theta_s, N_s):
    w.observe(update_euler, "value")
update_euler()
W.VBox([W.HBox([theta_s, N_s]), info2, fig_euler])

**考えてみよう**

- $\theta=\pi$ で $N$ を上げていくと、部分和は $-1$ に近づきます。$N$ が小さいうちは単位円から大きく外れるのに、最後は円の上の点に収まります。
- $\theta$ を $0$ から動かすと、ゴール $e^{i\theta}$ は単位円の上を回ります。$e^{i\theta}$ は「回転」を表し、[複素数の幾何学のノート](../../../研究ノート/04_群論・代数/complex_numbers_geometry.md)の Part II・IV で扱った「複素数の掛け算＝回転」とつながります。次のノートブックでは、これを行列で書きます。

## 5. テイラー展開できない「なめらかな関数」

$$
f(x)=\begin{cases}e^{-1/x^2}&(x\ne0)\\0&(x=0)\end{cases}
$$

は、何回でも微分できますが、$x=0$ での微分係数が**すべて $0$** です。そのため、マクローリン展開はすべての項が $0$ で、展開した級数は $0$ という関数になります。ところが $f(x)$ 自身は $x\neq0$ で $0$ ではありません。**「何回でも微分できる」だけでは、テイラー展開が元の関数に一致するとは限らない**という例です（一致する関数を**解析的**な関数と呼びます）。

In [ ]:
import sympy as sp

xsym = sp.symbols("x", real=True)
f = sp.exp(-1 / xsym**2)
for k in range(1, 5):
    dk = sp.diff(f, xsym, k)
    assert sp.limit(dk, xsym, 0) == 0                 # k 回微分の、x → 0 での極限は 0
print("1〜4 回微分の、x = 0 での値（極限）はすべて 0")

xs5 = np.linspace(-2, 2, 400)
with np.errstate(divide="ignore"):
    ys5 = np.where(xs5 == 0, 0.0, np.exp(-1 / xs5**2))
fig = go.Figure([go.Scatter(x=xs5.tolist(), y=ys5.tolist(), mode="lines", name="e^(−1/x²)"),
                 go.Scatter(x=xs5.tolist(), y=[0.0] * len(xs5), mode="lines", line=dict(dash="dash"), name="マクローリン展開（すべての項が 0）")])
fig.update_layout(height=340, width=760, xaxis_title="x", margin=dict(t=20))
fig

## 6. 丸め誤差：級数をそのまま足すと、答えが崩れる

$e^{-x}=\sum\frac{(-x)^k}{k!}$ を、$x=20$ でそのまま足してみます。正しい値は $e^{-20}\approx2.06\times10^{-9}$ です。

In [ ]:
def exp_series_naive(x: float, N: int = 150) -> float:
    total, term = 0.0, 1.0
    for k in range(N):
        total += term
        term *= x / (k + 1)
    return total


rows = []
for x in (1, 5, 10, 15, 20, 25):
    naive = exp_series_naive(-x)                     # e^(−x) の級数をそのまま足す
    better = 1 / exp_series_naive(x)                 # e^(x) を足してから逆数を取る
    true = np.exp(-x)
    rows.append((x, true, naive, abs(naive - true) / true, abs(better - true) / true))
    print(f"x = {x:2d}：正しい値 {true:.3e}、そのまま {naive:+.3e}（相対誤差 {abs(naive - true) / true:.1e}）、"
          f"逆数 {better:.3e}（相対誤差 {abs(better - true) / true:.1e}）")
assert rows[4][3] > 0.1 and rows[4][4] < 1e-12       # x = 20：そのままでは崩れ、逆数なら正確

terms = np.array([20.0**k / factorial(k) for k in range(80)])
fig = make_subplots(rows=1, cols=2, subplot_titles=("x と相対誤差", "x = 20 の各項の大きさ |(−20)^k / k!|"))
X = [r[0] for r in rows]
fig.add_trace(go.Scatter(x=X, y=[r[3] for r in rows], mode="lines+markers", name="そのまま足す"), 1, 1)
fig.add_trace(go.Scatter(x=X, y=[max(r[4], 1e-17) for r in rows], mode="lines+markers", name="e^x を足して逆数（0 は 10⁻¹⁷ で表示）"), 1, 1)
fig.add_trace(go.Scatter(x=list(range(80)), y=terms.tolist(), mode="lines", name="各項の大きさ"), 1, 2)
fig.add_trace(go.Scatter(x=[0, 79], y=[np.exp(-20)] * 2, mode="lines", line=dict(dash="dot", color="#333"), name="答え e^(−20)"), 1, 2)
fig.update_yaxes(type="log")
fig.update_xaxes(title_text="x", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_layout(height=380, width=950, margin=dict(t=40))
fig

**理由**：$x=20$ では、項の大きさが $k=20$ 付近で約 $4\times10^{7}$ まで大きくなり、正負が交互に打ち消し合って、最後に $2\times10^{-9}$ が残ります。コンピュータの数（倍精度）は、それぞれの数の約 $10^{-16}$ 倍の誤差を含むので、$4\times10^{7}$ の項の誤差は約 $4\times10^{-9}$ で、答えそのものより大きくなります（**桁落ち**）。$e^{20}$ を計算すると、項はすべて正で打ち消し合いが起きないので正確で、その逆数を取れば正しい値が得られます。

数学的に正しい式でも、計算の順序によって、数値は大きく狂うことがあります。カオスのノートブックで見た「丸め誤差が拡大する」話の、もっと身近な例です。

## 観察のポイント（まとめ）

- テイラー展開は「点のまわりの、多項式による近似」で、次数を上げると近似の範囲が広がる。ただし、届く範囲（収束半径）は、複素平面での一番近い特異点で決まる。
- $e^x$ の級数に $i\theta$ を入れると、実部と虚部がそれぞれ $\cos$ と $\sin$ の級数になり、オイラーの公式が出る。
- 何回でも微分できても、テイラー展開が元の関数に一致するとは限らない。
- 正しい級数でも、足し方によっては丸め誤差で崩れる。

**次のノートブック**：[02_matrix_exponential.ipynb](02_matrix_exponential.ipynb) で、級数の $x$ に行列を入れます。